# 

In [ ]:
%load_ext autoreload
%autoreload 2

from report import *

In [ ]:
job_dict = {
    #"Baseline" : "../results/job_14649999"
    #"Baseline" : "../results/job_14657724", #ReLU
    #"Baseline" : "../results/job_15013579", #Leaky ReLU
    #"Baseline" : "../results/job_15019651", #Leaky ReLU Softplus
    #"Baseline" : "../results/job_15032242", #Leaky ReLU Softplus SLOW
    #"Baseline" : "../results/job_15075017", #Reverted to better likelihood calculation
    #"Baseline": "../results/job_15086159", #Less kl and more adabelief
    #"Baseline": "../results/job_15430938", #Golden parameters
    #"Baseline": "../results/job_15444520", #Golden parameters
    #"Baseline": "../results/job_15466102", #oof do over
    "Baseline": "../results/job_15958064", 
    #"More KL": "../results/job_15958110", #kl=1e1
    #"Student" : "../results/job_15983920", #Student's t = 32
}
jr = JobReport(*job_dict.values(), names=list(job_dict.keys()))

# benchmarks for which we have a baseline 
baseline = jr.results[jr.results.name=='Baseline']
filtered = jr.results[jr.results.Benchmark.isin(baseline.Benchmark)]

# Exclude multi-wilson. you can comment this out to include
filtered = filtered[~filtered.Benchmark.str.contains('multi_wilson')]


In [ ]:
plt.figure()
plot_data = filtered
keys = ['Job Name'] + [k for k in plot_data if 'CC' in k]
sns.violinplot(
    plot_data[keys].melt('Job Name'),
    hue = 'Job Name',
    x = 'variable',
    y = 'value',
    inner='point',
    palette='Dark2',
)
plt.grid(ls='-.')
plt.xticks(rotation=45, rotation_mode='anchor', ha='right')
sns.move_legend(plt.gca(), "upper left", bbox_to_anchor=(1, 1))

plt.figure()
plot_data = filtered
keys = ['Job Name'] + [k for k in plot_data if 'NLL' in k]
sns.violinplot(
    plot_data[keys].melt('Job Name'),
    hue = 'Job Name',
    x = 'variable',
    y = 'value',
    inner='point',
    palette='Dark2',
)
plt.grid(ls='-.')
plt.xticks(rotation=45, rotation_mode='anchor', ha='right')
sns.move_legend(plt.gca(), "upper left", bbox_to_anchor=(1, 1))


plt.figure()
keys = ['Job Name'] + [k for k in plot_data if k.startswith('Rwork') or k.startswith('Rfree')]
sns.violinplot(
    plot_data[keys].melt('Job Name'),
    hue = 'Job Name',
    x = 'variable',
    y = 'value',
    inner='point',
    palette='Dark2',
)
plt.grid(ls='-.')
plt.xticks(rotation=45, rotation_mode='anchor', ha='right')
sns.move_legend(plt.gca(), "upper left", bbox_to_anchor=(1, 1))


plt.figure()
keys = ['Job Name'] + [k for k in plot_data if 'Peak' in k]
sns.violinplot(
    plot_data[keys].melt('Job Name'),
    hue = 'Job Name',
    x = 'variable',
    y = 'value',
    inner='point',
    palette='Dark2',
)
plt.ylabel("Anomalous Peak Height ($\sigma$)")
plt.grid(ls='-.')
plt.xticks(rotation=45, rotation_mode='anchor', ha='right')
plt.ylim(0, plt.ylim()[1])
sns.move_legend(plt.gca(), "upper left", bbox_to_anchor=(1, 1))

In [ ]:
xkey = 'Benchmark'
ykey = 'Peak Max'
lkey = 'Job Name'

literature_csv = f""""{lkey}",Benchmark,"{ykey}"
Published,cxidb_61,20.6
Published,cxidb_62,24.6
Published,cxidb_81,67.0
Published,cxidb_81_small,20.42
Published,hewl,19.89"""
from io import StringIO
df = pd.concat((
    jr.results,
    pd.read_csv(StringIO(literature_csv)),
))

sns.barplot(
    df.melt((xkey, lkey), value_vars=ykey).sort_values(xkey),
    x=xkey,
    y='value',
    hue=lkey,
    palette='Dark2',
)
plt.ylabel(ykey)
plt.grid(which='both', axis='y', ls='-.')

In [ ]:
for br in jr.reports:
    br._do_plots()

In [ ]:
literature_peak_max = {
    'cxidb_61' : 20.6,
    'cxidb_62' : 24.6,
    'cxidb_81' : 67.0,
    'cxidb_81_small' : 20.42,
    'hewl' : 19.89, #Aimless
    #'HEWL' : 16.13, #Careless without transfer learning
    #'HEWL' : 20.48, #Careless with transfer learning
} 

records = []
for _,row in jr.results.iterrows():
    k = row['Benchmark']
    if k in literature_peak_max:
        row['Published Peak'] = literature_peak_max[k]
        records.append(row)

df = pd.DataFrame.from_records(records)
df[r'Peak Change ($\sigma$)'] = df['Peak Final'] - df['Published Peak']
df[r'Peak Change (%)'] = 100.*(df['Peak Final'] - df['Published Peak']) / df['Published Peak']
df


In [ ]:
xkey = 'Coherent X-ray Imaging Data Bank Entry'
legend_key = 'Analysis'
ykey = r"Anomalous Peak Height ($\sigma$)"
published_name = "Published"
abismal_name = "Ours"
df[abismal_name] = df['Peak Final']
df[published_name] = df['Published Peak']
name_dict = {
    'cxidb_61' : 'CXIDB: 61',
    'cxidb_62' : 'CXIDB: 62',
    'cxidb_81' : 'CXIDB: 81',
    'cxidb_81_small' : 'CXIDB: 81\n(small)',
    'hewl' : 'HEWL\n(NE-CAT)',
}

df_selection = df[df['Benchmark'].isin(name_dict)]
df_selection[xkey] = df_selection['Benchmark'].apply(lambda x: name_dict[x])

plt.figure(figsize=(4.0, 3.0), dpi=150)
ax = sns.barplot(
    df_selection[[xkey, abismal_name, published_name]].melt(xkey, var_name=legend_key, value_name=ykey).sort_values(xkey),
    x = xkey,
    y = ykey,
    hue=legend_key,
    palette='Greys',
)
plt.xticks(*plt.xticks(), rotation=45, ha='right', rotation_mode='anchor')
_,ymax = plt.ylim()
yticks = np.arange(0, ymax, 10)
plt.yticks(yticks, minor=True)
plt.grid(which='both', ls='-.', axis='y')
# see: https://stackoverflow.com/questions/31506361/grid-zorder-seems-not-to-take-effect-matplotlib
ax.set_axisbelow(True)
plt.tight_layout()

In [ ]:
ykey='Anomalous Peak Height'
huekey = 'Benchmark Name'

include_benchmarks = [
    'cxidb_61',
    'cxidb_62',
    'cxidb_81',
    'cxidb_81_small',
    'hewl',
]

data = []
for br in jr.reports:
    df = br.history.join(
        br.peak_data[['Epoch', 'value']].groupby('Epoch').max().rename(columns={'value': ykey})
    )
    df[huekey] = br.summary['Benchmark']
    data.append(df)
df = pd.concat(data)
if include_benchmarks is not None:
    df = df[df[huekey].isin(include_benchmarks)]

df['Time (min)'] = df['Time (s)'] / 60.
xkey = "Time (min)"
ykey = "% Max Signal"
df[ykey] = 100. * df["Anomalous Peak Height"].to_numpy() / df[["Anomalous Peak Height", "Benchmark Name"]].groupby("Benchmark Name").transform(np.nanmax).to_numpy().flatten()
plt.figure(figsize=(4.0, 3.0), dpi=150)
data = df.sort_values(huekey)[['Epoch', xkey, ykey, huekey]].reset_index()
sns.lineplot(data, x=xkey, y=ykey, hue=huekey, palette='Dark2')
plt.grid(which='both', axis='both', ls='-.')
plt.xlim(0, 20.)

plt.figure(figsize=(4.0, 3.0), dpi=150)
sns.lineplot(df.dropna().sort_values(huekey), x='Epoch', y=ykey, hue=huekey, palette='Dark2')
plt.grid(which='both', axis='both', ls='-.')
#plt.xlim(1, 30)

df['Time (min)'] = df['Time (s)'] / 60.
ykey = "Time (min)"
plt.figure(figsize=(4.0, 3.0), dpi=150)
sns.lineplot(df.sort_values(huekey), x='Epoch', y=ykey, hue=huekey, palette='Dark2')
plt.grid(which='both', axis='both', ls='-.')
#plt.xlim(0, 30)

df['GPU Memory Usage (MB)'] = df['FB Used (MiB)']
df[[huekey, "GPU Memory Usage (MB)"]].groupby(huekey).max()